# CS13 Alignment Parameter Comparison

Compare the historical non-rigid parameter settings, inspect the selected result in 3D, and evaluate reference downsampling.

This curated notebook targets the current Dynamo-free Spateo API. Edit the configuration cell before execution.


## Configure the CS13 parameter experiment


In [ ]:
import os
import warnings
from pathlib import Path

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import numpy as np
import spateo as st
import torch

warnings.filterwarnings("ignore")
DEVICE = "0" if torch.cuda.is_available() else "cpu"
print(f"Spateo {st.__version__}; alignment device: {DEVICE}")

import anndata as ad

SLICE_PATHS = [
    Path("/DATA/User/gaomohan/DATA/CS13_Project/cs13/add_celltype/adata_109_processed.h5ad"),
    Path("/DATA/User/gaomohan/DATA/CS13_Project/cs13/add_celltype/adata_118_processed.h5ad"),
]
SPATIAL_KEY = "spatial"
ANNOTATION_KEY = "celltype"
ALIGN_KEY = "align_spatial"
Z_VALUES = [0.0, 40.0]
PARAMETER_GRID = {
    "iter500_k100": {"max_iter": 500, "K": 100},
    "iter500_k300": {"max_iter": 500, "K": 300},
    "iter300_k100": {"max_iter": 300, "K": 100},
}
SELECTED_TRIAL = "iter500_k100"


## Load, validate, and preprocess the CS13 pair


In [ ]:
slices = [st.read_h5ad(path) for path in SLICE_PATHS]
for path, adata in zip(SLICE_PATHS, slices):
    if SPATIAL_KEY not in adata.obsm or np.asarray(adata.obsm[SPATIAL_KEY]).shape[1] != 2:
        raise ValueError(f"{path.name} requires two-dimensional spatial coordinates.")
    if ANNOTATION_KEY not in adata.obs:
        raise KeyError(f"{path.name} is missing obs[{ANNOTATION_KEY!r}].")


def preprocess_slice(adata):
    adata = st.pp.filter_cells(adata, min_expr_genes=10, inplace=False)
    adata = st.pp.filter_genes(adata, min_cells=3, inplace=False)
    if "counts_X" not in adata.layers:
        source_layer = next(
            (candidate for candidate in ("counts", "raw_counts") if candidate in adata.layers),
            None,
        )
        if source_layer is None:
            warnings.warn("No count layer found; treating X as counts. Verify this assumption.")
        adata.layers["counts_X"] = (
            adata.layers[source_layer].copy() if source_layer is not None else adata.X.copy()
        )
    st.pp.normalize_total(
        adata,
        layer="counts_X",
        out_layer="norm_X",
        target_sum=None,
        size_factor_key="Size_Factor",
        inplace=True,
    )
    st.pp.log1p_layer(
        adata,
        layer="norm_X",
        out_layer="log1p_X",
        set_X=True,
        inplace=True,
    )
    return adata


slices = [preprocess_slice(adata) for adata in slices]
st.align.group_pca(slices, pca_key="X_pca", use_hvg=False)


## Review the unaligned pair


In [ ]:
st.pl.overlay_slices_2d(
    slices=slices,
    label_key=ANNOTATION_KEY,
    spatial_key=SPATIAL_KEY,
    height=3,
    overlay_type="backward",
    show_legend=True,
    axis_off=True,
)


## Compare non-rigid parameter settings


In [ ]:
alignment_trials = {}
mapping_trials = {}
for trial_name, trial_parameters in PARAMETER_GRID.items():
    aligned_pair, mapping = st.align.morpho_align(
        models=[adata.copy() for adata in slices],
        rep_layer="X_pca",
        rep_field="obsm",
        dissimilarity="cos",
        spatial_key=SPATIAL_KEY,
        key_added=ALIGN_KEY,
        device=DEVICE,
        verbose=True,
        beta=1,
        lambdaVF=1,
        **trial_parameters,
    )
    alignment_trials[trial_name] = aligned_pair
    mapping_trials[trial_name] = mapping


In [ ]:
for trial_name, aligned_pair in alignment_trials.items():
    print(trial_name, PARAMETER_GRID[trial_name])
    st.pl.overlay_slices_2d(
        slices=aligned_pair,
        spatial_key=f"{ALIGN_KEY}_nonrigid",
        height=3,
        overlay_type="backward",
        show_legend=False,
    )


## Stack the selected aligned pair in 3D


In [ ]:
selected_slices = [adata.copy() for adata in alignment_trials[SELECTED_TRIAL]]
for adata, z_value in zip(selected_slices, Z_VALUES):
    xy = np.asarray(adata.obsm[f"{ALIGN_KEY}_nonrigid"])
    adata.obs["z_height"] = z_value
    adata.obsm["aligned_spatial_3d"] = np.column_stack([xy, np.full(adata.n_obs, z_value)])

aligned_adata = ad.concat(selected_slices, label="slice_id", keys=["slice_109", "slice_118"])
aligned_pc, palette = st.tdr.construct_pc(
    adata=aligned_adata,
    spatial_key="aligned_spatial_3d",
    groupby=ANNOTATION_KEY,
    key_added="tissue",
    colormap="tab20",
)
st.pl.three_d_plot(
    model=aligned_pc,
    key="tissue",
    model_style="points",
    model_size=4,
    colormap=palette,
    show_axes=True,
    jupyter="static",
    window_size=(1200, 1200),
)


## Compare full-data and 10,000-cell reference alignment


In [ ]:
selected_parameters = PARAMETER_GRID[SELECTED_TRIAL]
aligned_down, aligned_reference, _, _ = st.align.morpho_align_ref(
    models=[adata.copy() for adata in slices],
    n_sampling=10000,
    sampling_method="random",
    rep_layer="X_pca",
    rep_field="obsm",
    dissimilarity="cos",
    spatial_key=SPATIAL_KEY,
    key_added=ALIGN_KEY,
    device=DEVICE,
    verbose=True,
    beta=1,
    lambdaVF=1,
    **selected_parameters,
)

st.pl.overlay_slices_2d(
    slices=aligned_down,
    spatial_key=f"{ALIGN_KEY}_nonrigid",
    height=3,
    overlay_type="backward",
    show_legend=False,
)
